# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [10]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias

from sklearn.manifold import TSNE
from tsfresh import extract_features, select_features, extract_relevant_features
from tsfresh.feature_extraction.settings import EfficientFCParameters, MinimalFCParameters, IndexBasedFCParameters
from tsfresh.utilities.dataframe_functions import impute

BASE_PATH = "../../.."
DATASET = f"{BASE_PATH}/stressid-dataset"
DATA_ECG = f"{DATASET}/ecg_windowed.csv"
DATA_EDA = f"{DATASET}/eda_windowed.csv"
LABELS_SEPARATOR = ","
LABELS = f"{BASE_PATH}/stressID/labels.csv"
DATA_SEPARATOR = ","
DATA_FS = 500  # Hz
DATA_WINDOW_DURATION = 60  # seconds
TARGET_FS = 51.2
RANDOM_STATE = 21

EDAECG_RELEVANT_FEAT_FILENAME = "../Features/tfresh_edacga_selfeatures.efficient.csv"
EDA_RELEVANT_FEAT_FILENAME = "../Features/tfresh_eda_selfeatures.efficient.csv"
ECG_RELEVANT_FEAT_FILENAME = "../Features/tfresh_ecg_selfeatures.efficient.csv"

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": False,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": False,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}

tasks_to_drop = [
    "2ea4_Counting1",
    "2ea4_Counting2",
    "2ea4_Reading",
    "2ea4_Speaking",
    "5f7t_Baseline",
    "5f7t_Counting1",
    "5f7t_Counting2",
    "5f7t_Counting3",
    "5f7t_Reading",
    "5f7t_Video2",
    "71i5_Counting1",
    "71i5_Video1",
    "dmbd_Counting1",
    "dmbd_Counting3",
    "dmbd_Math",
    "iqyg_Baseline",
    "iqyg_Counting1",
    "iqyg_Counting2",
    "iqyg_Reading",
    "iqyg_Speaking",
    "iqyg_Video2",
    "j1u8_Breathing",
    "r5s8_Counting3",
    "r5s8_Math",
    "r5s8_Speaking",
    "r5s8_Stroop",
    "tmvd_Relax",
]

@dataclass
class Dataset:
    X: list[pd.Series]
    y: pd.Series
    groups: np.ndarray[int]


CWT: TypeAlias = Tuple[np.ndarray[tuple[int], np.dtype], np.ndarray]

### Build dataset from data files

In [ ]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
labels: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        labels[key] = labels_df[conf["col_name"]]

display(labels["b"])

for to_drop in tasks_to_drop:
    labels["b"].drop(to_drop, inplace=True, errors="ignore")
    #labels["t"].drop(to_drop, inplace=True, errors="ignore")
    #labels["q"].drop(to_drop, inplace=True, errors="ignore")

display(labels["b"])


subject/task
2ea4_Breathing    0
2ea4_Counting1    1
2ea4_Counting2    1
2ea4_Counting3    1
2ea4_Math         1
                 ..
y9z6_Relax        0
y9z6_Speaking     1
y9z6_Stroop       1
y9z6_Video1       1
y9z6_Video2       0
Name: binary-stress, Length: 700, dtype: int64

subject/task
2ea4_Breathing    0
2ea4_Counting3    1
2ea4_Math         1
2ea4_Relax        0
2ea4_Stroop       0
                 ..
y9z6_Relax        0
y9z6_Speaking     1
y9z6_Stroop       1
y9z6_Video1       1
y9z6_Video2       0
Name: binary-stress, Length: 675, dtype: int64

In [18]:
# Pairing labels and samples for each class type

raw_num_samples = DATA_FS * DATA_WINDOW_DURATION
raw_eda = pd.read_csv(DATA_EDA)
raw_ecg = pd.read_csv(DATA_ECG)

subjects_in_columns: list[int] = [] # subjects (should be 699 after joining eda and ecg)
subject_series: list[int] = []  # "Subject"
task_series: list[str] = []  # "Task"
sample_n_series: list[int] = []  # "Nth-Sample"
eda_series: list[float] = []  # "EDA"
ecg_series: list[float] = []  # "ECG"

blabel_series: list[int] = []  # "b-class"
tlabel_series: list[int] = []  # "t-class"
qlabel_series: list[int] = []  # "q-class"

for class_type, labels_set in labels.items():
    subject_to_group: dict[str, int] = {}
    group_counter = 0
    subjects_list: list[int] = []

    for col_name, label in labels_set.items():
        subject_id = col_name.split("_")[0]
        if (
            col_name in raw_eda.columns
        ):  # Only add labels with corresponding data, EDA dataset lacks one entry which ECG has
            length = raw_eda[col_name].size
            sample_n_series.extend([n for n in range(length)])
            eda_series.extend(raw_eda[col_name])
            ecg_series.extend(raw_ecg[col_name])
            label_values = np.full(length, label)
            if class_type == "b":
                blabel_series.extend(label_values)
            elif class_type == "t":
                tlabel_series.extend(label_values)
            elif class_type == "q":
                qlabel_series.extend(label_values)
            if subject_id not in subject_to_group:
                subject_to_group[subject_id] = group_counter
                group_counter += 1
            subject_series.extend(np.full(length, subject_to_group[subject_id]))
            subjects_in_columns.append(subject_to_group[subject_id])
            task_series.extend(np.full(length, col_name))

data_df = pd.DataFrame({
    # "Subject": subject_series,
    "Task": task_series,
    "Nth-Sample": sample_n_series,
    "EDA": eda_series,
    "ECG": ecg_series})
display(data_df)
only_eda_df = data_df.drop(columns="ECG")
display(only_eda_df)
only_ecg_df = data_df.drop(columns="EDA")
display(only_ecg_df)

tasks_labels = labels["b"].copy()

,Task,Nth-Sample,EDA,ECG
0,2ea4_Breathing,0,32019.942317,10398.583885
1,2ea4_Breathing,1,32019.783775,9831.512610
2,2ea4_Breathing,2,32019.625308,8892.997782
3,2ea4_Breathing,3,32019.466918,7709.957079
4,2ea4_Breathing,4,32019.308609,6422.639589
...,...,...,...,...
20249995,y9z6_Video2,29995,4304.461086,1.998284
20249996,y9z6_Video2,29996,4304.435349,8.936071
20249997,y9z6_Video2,29997,4304.409994,9.970963
20249998,y9z6_Video2,29998,4304.385020,5.986312


,Task,Nth-Sample,EDA
0,2ea4_Breathing,0,32019.942317
1,2ea4_Breathing,1,32019.783775
2,2ea4_Breathing,2,32019.625308
3,2ea4_Breathing,3,32019.466918
4,2ea4_Breathing,4,32019.308609
...,...,...,...
20249995,y9z6_Video2,29995,4304.461086
20249996,y9z6_Video2,29996,4304.435349
20249997,y9z6_Video2,29997,4304.409994
20249998,y9z6_Video2,29998,4304.385020


,Task,Nth-Sample,ECG
0,2ea4_Breathing,0,10398.583885
1,2ea4_Breathing,1,9831.512610
2,2ea4_Breathing,2,8892.997782
3,2ea4_Breathing,3,7709.957079
4,2ea4_Breathing,4,6422.639589
...,...,...,...
20249995,y9z6_Video2,29995,1.998284
20249996,y9z6_Video2,29996,8.936071
20249997,y9z6_Video2,29997,9.970963
20249998,y9z6_Video2,29998,5.986312


In [23]:
# BOTH EDA and ECG
EXTRACT_EDAECG = False
if EXTRACT_EDAECG:
    X = extract_relevant_features(
        data_df,
        y=tasks_labels,
        column_id="Task",
        column_sort="Nth-Sample",
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [22]:
if EXTRACT_EDAECG:
    X.to_csv(EDAECG_RELEVANT_FEAT_FILENAME)
else:
    X = pd.read_csv(EDAECG_RELEVANT_FEAT_FILENAME, index_col=0)
display(X)


,"ECG__fft_coefficient__attr_""abs""__coeff_0","ECG__fft_aggregated__aggtype_""skew""","ECG__fft_aggregated__aggtype_""kurtosis""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""mean""","EDA__linear_trend__attr_""stderr""",...,EDA__ar_coefficient__coeff_5__k_10,"ECG__fft_coefficient__attr_""abs""__coeff_55","EDA__fft_coefficient__attr_""angle""__coeff_48","ECG__change_quantiles__f_agg_""mean""__isabs_False__qh_0.8__ql_0.4","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.0","ECG__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","EDA__fft_coefficient__attr_""abs""__coeff_6","EDA__fft_coefficient__attr_""imag""__coeff_51","EDA__fft_coefficient__attr_""imag""__coeff_20","EDA__fft_coefficient__attr_""angle""__coeff_23"
2ea4_Breathing,3.856781e+04,3.535058,17.045377,0.117937,0.010721,0.003798,0.003804,0.010761,0.120433,0.000340,...,1.599525,479494.836534,-24.063132,0.057408,44.846211,0.023982,2.561508e+06,6671.000929,1.013796e+05,-152.901779
2ea4_Counting3,2.296474e-09,4.340321,59.903030,0.445783,0.039885,0.014101,0.014103,0.039895,0.446500,0.001261,...,-1.562988,364764.289081,-75.561782,-0.088318,43.184946,0.025476,1.659211e+07,-868304.646636,-1.589312e+06,7.856022
2ea4_Math,1.413355e-09,5.474369,69.845463,0.528900,0.047481,0.016794,0.016802,0.047530,0.531978,0.001503,...,-1.798798,328808.454714,57.448704,-0.006699,36.294536,0.025517,8.606846e+06,-115591.443437,-4.890181e+05,114.766269
2ea4_Relax,8.458034e+03,4.849032,67.736627,0.103256,0.009338,0.003306,0.003310,0.009364,0.104811,0.000296,...,1.599486,507141.928489,-165.606244,0.082743,28.422107,0.023976,2.637859e+06,-17572.263861,-4.271464e+05,-33.806173
2ea4_Stroop,1.607987e-09,1.584479,22.606167,0.719616,0.064550,0.022829,0.022838,0.064607,0.723157,0.002042,...,-1.517883,741381.956918,84.733488,0.244575,35.826190,0.024708,1.167900e+07,686169.397828,-2.863449e+06,69.584787
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
y9z6_Relax,3.728582e+03,2.275315,39.993216,0.086170,0.007697,0.002721,0.002721,0.007697,0.086164,0.000243,...,1.598975,138985.310833,-86.539270,-0.684143,31.331148,0.011253,1.876242e+06,-109111.477259,-2.815163e+05,-88.141079
y9z6_Speaking,6.730261e-10,1.567183,24.260312,0.325079,0.028989,0.010245,0.010244,0.028978,0.324404,0.000916,...,-1.505806,481023.895035,91.431465,-0.719476,48.419936,0.015300,4.420559e+06,294496.084877,8.306524e+05,106.232817
y9z6_Stroop,2.055458e-09,1.668339,28.279137,0.206460,0.018324,0.006472,0.006468,0.018297,0.204817,0.000578,...,-1.465195,109133.052717,101.297205,-0.710273,34.434699,0.012290,4.023467e+06,96294.400595,3.642958e+05,4.787798
y9z6_Video1,2.153411e+03,2.881601,45.335105,0.213687,0.019081,0.006745,0.006745,0.019080,0.213599,0.000603,...,-1.579590,375250.952270,-90.123967,-0.436634,25.770521,0.010987,2.388052e+06,-103200.738898,-2.121321e+05,-81.483762


In [27]:
EXTRACT_EDA = False
if EXTRACT_EDA:
    eda_rel_features = extract_relevant_features(
        only_eda_df,
        y=tasks_labels,
        column_id="Task",
        column_sort="Nth-Sample",
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [26]:
if EXTRACT_EDA:
    eda_rel_features.to_csv(EDA_RELEVANT_FEAT_FILENAME)
else:
    eda_rel_features = pd.read_csv(EDA_RELEVANT_FEAT_FILENAME, index_col=0)
display(eda_rel_features)


,"EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""mean""","EDA__linear_trend__attr_""stderr""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""max""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""max""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""max""",...,"EDA__agg_linear_trend__attr_""rvalue""__chunk_len_5__f_agg_""mean""","EDA__agg_linear_trend__attr_""rvalue""__chunk_len_10__f_agg_""max""","EDA__fft_coefficient__attr_""imag""__coeff_16","EDA__agg_linear_trend__attr_""rvalue""__chunk_len_50__f_agg_""max""","EDA__fft_coefficient__attr_""imag""__coeff_38","EDA__fft_coefficient__attr_""real""__coeff_71",EDA__mean_second_derivative_central,"EDA__fft_coefficient__attr_""abs""__coeff_44","EDA__fft_coefficient__attr_""angle""__coeff_25","EDA__fft_coefficient__attr_""angle""__coeff_26"
2ea4_Breathing,0.117937,0.010721,0.003798,0.003804,0.010761,0.120433,0.000340,0.010802,0.003810,0.122850,...,0.138115,0.140014,2.792524e+05,0.148320,1.235367e+04,-6801.159984,-0.000026,116254.940245,-24.629819,-116.216706
2ea4_Counting3,0.445783,0.039885,0.014101,0.014103,0.039895,0.446500,0.001261,0.039905,0.014104,0.446998,...,-0.875228,-0.875443,2.353181e+06,-0.876485,-1.007332e+06,6096.530913,0.000004,876953.190835,-128.128051,116.644807
2ea4_Math,0.528900,0.047481,0.016794,0.016802,0.047530,0.531978,0.001503,0.047579,0.016809,0.534843,...,-0.721519,-0.721839,-2.590433e+05,-0.723384,6.665997e+05,-7362.528193,0.000006,220947.779142,56.522580,-9.685534
2ea4_Relax,0.103256,0.009338,0.003306,0.003310,0.009364,0.104811,0.000296,0.009389,0.003314,0.106358,...,-0.029398,-0.029195,-2.816289e+05,-0.028313,-2.866168e+04,-736.591837,0.000002,19732.391205,-141.470132,-12.295197
2ea4_Stroop,0.719616,0.064550,0.022829,0.022838,0.064607,0.723157,0.002042,0.064663,0.022847,0.726481,...,-0.492318,-0.492156,9.102576e+05,-0.491553,9.151151e+05,7751.933432,0.000020,668933.175898,130.866405,79.405578
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
y9z6_Relax,0.086170,0.007697,0.002721,0.002721,0.007697,0.086164,0.000243,0.007696,0.002721,0.086156,...,-0.447275,-0.447405,-4.030830e+05,-0.447992,-1.485470e+05,1779.210774,0.000003,123188.219766,-95.418127,-93.348028
y9z6_Speaking,0.325079,0.028989,0.010245,0.010244,0.028978,0.324404,0.000916,0.028967,0.010242,0.323709,...,-0.035494,-0.036207,1.475436e+06,-0.039384,4.012580e+05,-8877.128652,-0.000010,316816.824593,82.103572,96.493705
y9z6_Stroop,0.206460,0.018324,0.006472,0.006468,0.018297,0.204817,0.000578,0.018269,0.006463,0.203152,...,-0.868126,-0.868562,3.325908e+05,-0.870518,1.464336e+05,-6125.258797,-0.000011,118682.499077,-3.981411,67.113676
y9z6_Video1,0.213687,0.019081,0.006745,0.006745,0.019080,0.213599,0.000603,0.019078,0.006744,0.213508,...,-0.216401,-0.216448,-3.877089e+05,-0.216658,-1.242788e+05,3159.590890,0.000006,127367.641687,-73.361998,-87.011042


In [31]:
EXTRACT_ECG = False
if EXTRACT_ECG:
    ecg_rel_features = extract_relevant_features(
        only_ecg_df,
        y=tasks_labels,
        column_id="Task",
        column_sort="Nth-Sample",
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [30]:
if EXTRACT_ECG:
    ecg_rel_features.to_csv(ECG_RELEVANT_FEAT_FILENAME)
else:
    ecg_rel_features = pd.read_csv(ECG_RELEVANT_FEAT_FILENAME, index_col=0)
display(ecg_rel_features)


,"ECG__fft_coefficient__attr_""abs""__coeff_0","ECG__fft_aggregated__aggtype_""skew""","ECG__fft_aggregated__aggtype_""kurtosis""","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.4","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.6__ql_0.4","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.2","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_0.6__ql_0.4",...,"ECG__change_quantiles__f_agg_""var""__isabs_False__qh_1.0__ql_0.4","ECG__fft_coefficient__attr_""abs""__coeff_20","ECG__fft_coefficient__attr_""abs""__coeff_78",ECG__lempel_ziv_complexity__bins_100,"ECG__cwt_coefficients__coeff_3__w_5__widths_(2, 5, 10, 20)",ECG__quantile__q_0.4,"ECG__fft_coefficient__attr_""abs""__coeff_11","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_1.0__ql_0.2","ECG__fft_coefficient__attr_""abs""__coeff_73","ECG__fft_coefficient__attr_""abs""__coeff_84"
2ea4_Breathing,3.856781e+04,3.535058,17.045377,25.792462,1504.356336,5080.609963,41.746862,3337.904080,40.620393,839.550168,...,53077.880990,71234.666532,762216.090730,0.154767,16046.180105,-109.733059,81531.156252,35273.039337,5.229027e+05,791698.975212
2ea4_Counting3,2.296474e-09,4.340321,59.903030,32.734921,1833.358520,4296.643855,43.621820,2393.780709,44.266107,762.268144,...,57133.649546,47460.311036,591110.218759,0.164867,1088.059955,-135.233547,30664.153547,37047.809452,1.156736e+06,282213.734293
2ea4_Math,1.413355e-09,5.474369,69.845463,26.163356,1281.345027,3461.666104,35.616231,2193.155822,36.250203,597.188630,...,58594.586597,37970.813199,886440.308747,0.149833,-7.504416,-125.604960,26539.294607,39043.490405,6.292091e+05,336679.253760
2ea4_Relax,8.458034e+03,4.849032,67.736627,20.488100,844.705346,2432.050492,27.622365,1669.063489,28.255120,425.215775,...,51147.230536,15074.832362,570605.319083,0.134833,-166.164547,-113.708470,8863.245338,35001.801993,5.074199e+05,234809.507478
2ea4_Stroop,1.607987e-09,1.584479,22.606167,27.257766,1324.136672,3142.518570,34.928442,1922.539259,36.019119,582.184370,...,54253.795403,96776.676500,439293.058232,0.151733,-551.501779,-127.192438,93201.915928,35873.358014,3.811128e+05,279495.467317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
y9z6_Relax,3.728582e+03,2.275315,39.993216,15.955197,472.849990,902.551862,21.374394,446.786247,21.525032,218.282910,...,19498.622878,44182.591440,209502.737616,0.159367,47.228377,-44.314598,5345.328828,13683.532872,1.411136e+05,262868.467023
y9z6_Speaking,6.730261e-10,1.567183,24.260312,24.835647,1169.412148,3254.926738,34.123537,2090.902166,32.418761,552.629907,...,25941.559426,105030.265526,839940.743385,0.171867,-340.859558,-128.859917,42909.588246,17599.518954,2.610783e+05,254074.960392
y9z6_Stroop,2.055458e-09,1.668339,28.279137,21.067075,830.347788,1477.314268,25.378243,834.390512,26.173005,386.568733,...,20062.023622,58416.089684,290008.053347,0.175067,-91.147754,-96.736188,47792.853094,13973.019192,3.279967e+05,263516.556613
y9z6_Video1,2.153411e+03,2.881601,45.335105,13.272262,324.117650,801.174301,18.141931,472.241567,17.785333,147.965477,...,15644.448358,27626.751630,341601.493876,0.157733,-111.075444,-74.327781,11891.727401,10788.616033,1.396112e+05,269302.124217


In [52]:
# Selecting rows that actually have entries in "labels" file
idx = list(X.merge(labels["b"], left_index=True, right_index=True).index)
y = labels["b"].loc[idx]
x = X.loc[idx]
print(f"Selected {len(x)} entries from X and {len(y)} labels.")

config = {
    "responsive": False,
    "toImageButtonOptions": {
        "format": "png",  # one of png, svg, jpeg, webp
        "height": 1200,
        "width": 1500,
        "scale": 1,  # Multiply title/legend/axis/canvas sizes by this factor
    },
}


Selected 675 entries from X and 675 labels.


In [70]:
# Divergence analysis on StressID
import plotly.express as px

perplexity = np.arange(20, 500, 20)
divergence = []
si_Ncomp = 2

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i, learning_rate=750.0)
    reduced = model.fit_transform(ecg_rel_features)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [54]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10) X
si_Ncomp = 2
si_Perp = 190
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(X)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y, width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_cls.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-ECGEDA-byclass"
fig_cls.show(config=config)

fig_subj = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=subjects_in_columns, width=800, height=600)
fig_subj.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_subj.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-ECGEDA-bysubject"
fig_subj.show(config=config)

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=n_subjects)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]
filtered_subjects_rows = [str(subject) for subject, flag in zip(subjects_in_columns, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(y.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID dataset t-SNE (TSFresh ECG+EDA features) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_subj.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-ECGEDA-by4subj"
fig_subj.show(config=config)

0.051395706832408905

In [35]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10) X
si_Ncomp = 2
si_Perp = 190
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=5)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)

si_X_tsne = si_tsne.fit_transform(X)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y.astype(str), width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_cls.show()

0.051395706832408905

In [59]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(20, 220, 20) eda_rel_features
si_Ncomp = 2
si_Perp = 190
BY_CLASS = True
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(eda_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y, width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh EDA features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_cls.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-EDA-byclass"
fig_cls.show(config=config)

fig_subj = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=subjects_in_columns, width=800, height=600)
fig_subj.update_layout(
    title="StressID dataset t-SNE (TSFresh EDA features) by Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_subj.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-EDA-bysubject"
fig_subj.show(config=config)

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=n_subjects)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]
filtered_subjects_rows = [str(subject) for subject, flag in zip(subjects_in_columns, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(y.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID dataset t-SNE (TSFresh EDA features) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_subj.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = f"tSNE-tsFresh-ECGEDA-by{n_subjects}subj"
fig_subj.show(config=config)


0.05107754468917847

In [71]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(20, 500, 20) ecg_rel_features
si_Ncomp = 2
si_Perp = 470
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(ecg_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y, width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_cls.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-ECG-byclass"
fig_cls.show(config=config)

fig_subj = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=subjects_in_columns, width=800, height=600)
fig_subj.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG features) by Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_subj.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = "tSNE-tsFresh-ECG-bysubject"
fig_subj.show(config=config)

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=n_subjects)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]
filtered_subjects_rows = [str(subject) for subject, flag in zip(subjects_in_columns, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(y.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID dataset t-SNE (TSFresh ECG features) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
    font=dict(size=22),
)
fig_subj.update_traces(marker=dict(size=28, opacity=0.7), selector=dict(mode="markers"))
config["toImageButtonOptions"]["filename"] = f"tSNE-tsFresh-ECG-by{n_subjects}subj"
fig_subj.show(config=config)


0.05004950240254402

In [43]:
# t-SNE in StressID
# Best N-comp=3, Perp=90 np.arange(15, 210, 15) X_selected
si_Ncomp = 3
si_Perp = 90
add_noise = False # For when the data points are too close together

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(X)
display(si_tsne.kl_divergence_)
for_display = si_X_tsne + np.random.normal(0, 0.01, si_X_tsne.shape) if add_noise else si_X_tsne

fig_cls = px.scatter_3d(
    x=for_display[:, 0], y=for_display[:, 1], z=for_display[:, 2], opacity=0.6, color=y, width=800, height=600
)
fig_cls.update_layout(title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Class")
fig_cls.show()
fig_subj = px.scatter_3d(
    x=for_display[:, 0],
    y=for_display[:, 1],
    z=for_display[:, 2],
    color=subjects_in_columns,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Subject")
fig_subj.show()

0.05290544033050537

In [50]:
# t-SNE in StressID
# Best N-comp=3, Perp=95 np.arange(5, 100, 5) eda_rel_features
si_Ncomp = 3
si_Perp = 95
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(eda_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter_3d(
    x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:, 2], color=y, opacity=0.5, width=800, height=600
)
fig_cls.update_layout(title="StressID dataset t-SNE (TSFresh EDA features) by Class")
fig_cls.show()
fig_subj = px.scatter_3d(
    x=si_X_tsne[:, 0],
    y=si_X_tsne[:, 1],
    z=si_X_tsne[:, 2],
    color=subjects_in_columns,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title="StressID dataset t-SNE (TSFresh EDA features) by Subject")
fig_subj.show()

0.05093998834490776

In [51]:
# t-SNE in StressID
# Best N-comp=3, Perp=95 np.arange(10, 150, 10) ecg_rel_features
si_Ncomp = 3
si_Perp = 100
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(ecg_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter_3d(
    x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:, 2], color=y, opacity=0.5, width=800, height=600
)
fig_cls.update_layout(title="StressID dataset t-SNE (TSFresh ECG features) by Class")
fig_cls.show()
fig_subj = px.scatter_3d(
    x=si_X_tsne[:, 0],
    y=si_X_tsne[:, 1],
    z=si_X_tsne[:, 2],
    color=subjects_in_columns,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title="StressID dataset t-SNE (TSFresh ECG features) by Subject")
fig_subj.show()


0.05462796613574028